# HARD & SOFT VOTING 

In [37]:
# IMPRORTING IMPORTANT ML LIBRARIES 

import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer 
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import VotingClassifier, VotingRegressor
from sklearn.svm import SVC
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score
import matplotlib.pyplot as plt

# Loading Dataset

In [18]:
df = pd.read_csv("human_cognitive_performance.csv")
df_sample = df.sample(n = 10000, random_state=42)   # As difficult to run 80K dataset with hard and sof voting, we need to get only 10K

In [19]:
df_sample.info()

<class 'pandas.DataFrame'>
Index: 10000 entries, 47044 to 58464
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   User_ID             10000 non-null  str    
 1   Age                 10000 non-null  int64  
 2   Gender              10000 non-null  str    
 3   Sleep_Duration      10000 non-null  float64
 4   Stress_Level        10000 non-null  int64  
 5   Diet_Type           10000 non-null  str    
 6   Daily_Screen_Time   10000 non-null  float64
 7   Exercise_Frequency  10000 non-null  str    
 8   Caffeine_Intake     10000 non-null  int64  
 9   Reaction_Time       10000 non-null  float64
 10  Memory_Test_Score   10000 non-null  int64  
 11  Cognitive_Score     10000 non-null  float64
 12  AI_Predicted_Score  10000 non-null  float64
dtypes: float64(5), int64(4), str(4)
memory usage: 1.1 MB


In [20]:
df_sample.nunique()

User_ID               10000
Age                      42
Gender                    3
Sleep_Duration           61
Stress_Level             10
Diet_Type                 3
Daily_Screen_Time       111
Exercise_Frequency        3
Caffeine_Intake         500
Reaction_Time          8792
Memory_Test_Score        60
Cognitive_Score        5672
AI_Predicted_Score     5791
dtype: int64

In [21]:
# Identifying tartget columns 
df_sample["Diet_Type"] = df_sample["Diet_Type"].map({
    "Vegetarian" : 0,
    "Non-Vegetarian" : 1,
    "Vegan" : 2
}) # both are metrics of brain activity and mental capacity.


In [24]:
target_col = "Diet_Type"

X = df_sample.drop(columns =["User_ID", "AI_Predicted_Score", target_col])
y = df_sample[target_col]

print("X shakli: ", X.shape)
print("y shakli: ", y.shape)

X shakli:  (10000, 10)
y shakli:  (10000,)


# Train/test split 

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [26]:
categorical_features = ["Gender","Exercise_Frequency"]
numerical_features = [col for col in X.columns if col not in categorical_features]

# an alternative way:
# categorical_features = X.select_dtypes(inlcude = ["int64", "float64"]).columns.tolist()
# numerical_features = X.select_dtypes(inlcude = ["object", "string"]).columns.tolist()

# Pipeline stage 

In [27]:
numerical_transformer = Pipeline(steps = [("scaler", StandardScaler())])
categorical_transformer = Pipeline(steps = [
    ("encoder", OneHotEncoder(handle_unknown = "ignore"))
])

# Preprocessor --> ColumnTransfomer

In [28]:
preprocessor = ColumnTransformer(
    transformers = [
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Hard Voting 

In [29]:
lr = LogisticRegression()
dt = DecisionTreeClassifier()
svm = SVC(probability = True)  # as Soft Voting adresses form probability results


In [30]:
hard_voting = VotingClassifier(
    estimators = [("lr", lr), ("dt", dt), ("svm", svm)],
    voting = "hard",
    n_jobs=-1
)         

In [31]:
pipeline_hard_voting = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("hard_voting", hard_voting)
    ]
)


In [14]:
pipeline_hard_voting 

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('hard_voting', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3

In [32]:
pipeline_hard_voting.fit(X_train, y_train)   

/Users/murodjongafforov/Desktop/mp2_8th_lesson/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('hard_voting', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['Age','Gender','Sleep_Duration',...,'Reaction_Time','Memory_Test_Score', 'Cognitive_Score']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``re

# Prediction of hard voting

In [33]:
y_pred_hard = pipeline_hard_voting.predict(X_test)
print("Hard Voting Accuracy:", accuracy_score(y_test, y_pred_hard))

Hard Voting Accuracy: 0.602


# Soft Voting 

In [35]:
soft_voting = VotingClassifier(
    estimators = [("lr", lr), ("dt", dt), ("svm", svm)],
    voting = "soft",
    n_jobs = -1
)

pipeline_soft_voting = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("soft_voting", soft_voting)
    ]
)
pipeline_soft_voting.fit(X_train, y_train)

y_pred_soft = pipeline_soft_voting.predict(X_test)
print("Soft Voting Accuracy: ", accuracy_score(y_test, y_pred_soft))
print("\n Soft Voting Predicted Probabilities(first 5 samples): ")
print(pipeline_soft_voting.predict_proba(X_test[:5]))


/Users/murodjongafforov/Desktop/mp2_8th_lesson/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Soft Voting Accuracy:  0.473

 Soft Voting Predicted Probabilities(first 5 samples): 
[[0.19615952 0.40683511 0.39700538]
 [0.21381866 0.72436065 0.06182069]
 [0.2039992  0.72436866 0.07163214]
 [0.20584741 0.73086445 0.06328814]
 [0.20097668 0.73608003 0.06294329]]


# Voting --> Regression 

In [38]:
lr_reg = LinearRegression()
ridge = Ridge()
rf = RandomForestRegressor()

In [40]:
# Voting Regresor
voting_reg = VotingRegressor(
    estimators = [("lr_reg", lr_reg), ("ridge", ridge), ("rf", rf)]
)

pipeline_voting_reg = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("voting_reg", voting_reg)
    ]
)
pipeline_voting_reg.fit(X_train, y_train)
y_pred_voting_reg = pipeline_voting_reg.predict(X_test)

In [43]:
print("Voting Regressor MSE: ", mean_squared_error(y_test, y_pred_voting_reg))
print("Voting Regressor R2: ", r2_score(y_test, y_pred_voting_reg))
print("\nVoting Regession Prediction(first 5 samples): ")
print(y_pred_voting_reg[:5])

Voting Regressor MSE:  0.3626262936398655
Voting Regressor R2:  -0.008460592350787444

Voting Regession Prediction(first 5 samples): 
[0.75606374 0.74974774 0.79788995 0.76958163 0.78011281]
